# ML Masterclass 05: Unsupervised & Manifold Learning

Welcome to Notebook 05. The dream of AI is algorithms that learn without human labels. In reality, Unsupervised Learning is notoriously difficult to evaluate and highly sensitive to hyperparameters.

We will explore Clustering (grouping similar things) and Dimensionality Reduction (compressing many features into few) while confronting the mathematical flaws of introductory algorithms.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"If you run Standard K-Means multiple times on the exact same dataset, you might get entirely different clusters. Why does this happen mathematically, and how exactly does the K-Means++ initialization trick fix it?"*

---

## Table of Contents
1. **Centroid Clustering:** Writing K-Means from scratch and understanding its spherical bias.
2. **Density Clustering:** DBSCAN and arbitrary shapes.
3. **Manifold Learning:** Why you should NEVER use t-SNE or UMAP as a preprocessing step for Logistic Regression.


## 1. Centroid Clustering (K-Means)

K-Means (Lloyd's Algorithm) is simple:
1. Pick $K$ random points as centroids.
2. Assign every data point to the nearest centroid.
3. Move the centroid to the mathematical mean of all points assigned to it.
4. Repeat until convergence.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does standard K-Means yield different results on the same data?*

**A:** Because the loss landscape (Inertia: sum of squared distances to nearest centroid) is highly **non-convex**. If your initial random centroids happen to spawn very close to each other, they will get trapped in a local minimum and slice a single true cluster in half. 
**K-Means++** fixes this by explicitly mathematically enforcing that the initial centroids are spawned as far away from each other as physically possible. This almost guarantees finding the true global minimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: K-Means from Scratch
# -----------------------------------------------------

X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

def custom_kmeans(X, k=4, max_iters=100):
    # 1. Randomly initialize centroids from existing points
    random_idx = np.random.choice(len(X), size=k, replace=False)
    centroids = X[random_idx]
    
    for _ in range(max_iters):
        # 2. Assign clusters based on nearest centroid (Vectorized Euclidean Distance)
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        
        # 3. Calculate new centroids
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        
        # Check for convergence
        if np.all(centroids == new_centroids):
            break
        centroids = new_centroids
        
    return labels, centroids

labels, centroids = custom_kmeans(X, k=4)

plt.figure(figsize=(8,5))
plt.scatter(X[:, 0], X[:, 1], c=labels, s=50, cmap='viridis', alpha=0.6)
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centroids')
plt.title("K-Means built from Scratch in NumPy")
plt.legend()
plt.show()

## 2. Density Clustering (DBSCAN)

K-Means assumes clusters are spherical/convex and requires you to guess $K$. 
What if the clusters are shaped like crescents or concentric rings?

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** does not require $K$. It groups points that are closely packed together (many neighbors within a radius `eps`) and marks points in low-density regions as outliers/noise.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *DBSCAN doesn't require K. What two hyperparameters MUST you set instead, and how do you tune them?*

**A:** You must set `eps` (the maximum distance between two samples for one to be considered in the neighborhood of the other) and `min_samples` (the number of samples in a neighborhood for a point to be considered a core point). You tune them using a **k-distance graph**—plotting the distance to every point's $k$-th nearest neighbor and looking for the "elbow" where the distance shoots up.

## 3. Manifold Learning (t-SNE & UMAP)

PCA (Principal Component Analysis) is a *linear* dimensionality reduction technique. It rotates data. If data is rolled up like a Swiss Roll, PCA just flattens it into a smashed mess.

**Manifold Learning** algorithms (like t-SNE and UMAP) "unroll" the Swiss Roll by preserving local neighborhood distances while ignoring global distances.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why is it mathematically invalid to use t-SNE or UMAP to reduce dimensions as a preprocessing step before training a classifier like Logistic Regression?*

**A:** 
1. **No Inference Function:** t-SNE physically moves the existing points around until they look nice. It DOES NOT learn a mathematical mapping formula $f(x) \rightarrow y$. Therefore, when a new, unseen point arrives in production, you cannot pass it through t-SNE to classify it.
2. **Global Distance Destruction:** t-SNE preserves *local* structure (who your neighbors are) but completely destroys *global* distances (how far cluster A is from cluster B). By feeding t-SNE coordinates into a model, you feed it fake spatial data. 
*Rule of Thumb: Manifold learning is ONLY for human visualization. PCA is for machine preprocessing.*